# Fine-tune a multilingual emotion model (LoRA) — Colab / Kaggle

**Status: not executed in the Manovia sandbox.** There is no GPU there and the
Hugging Face hub is blocked, so this notebook is the ready-to-run path for the
first *real* training run. The same code path was exercised on CPU with a
randomly-initialised tiny model (`configs/emotion_lora_smoke.yaml`, seconds), so
what is untested here is the *data and the scale*, not the training loop.

What it does, in order: check the accelerator → install → fetch GoEmotions
(Apache-2.0, catalogue § 1) → convert it to Manovia's nine emotions through the
documented mapping table → train `FacebookAI/xlm-roberta-base` (MIT) with LoRA
→ tune per-label thresholds on the validation split → evaluate on the
hand-written 174-row en/hi/bn set that is never trained on → package the
artefact → print exactly which files to bring back.

Rules this notebook obeys (AGENTS.md, ML ground rules): it is exploration, so
every decision it uses lives in `ml/src/` with a test; nothing here is quoted as
a result without an MLflow run id; **no rows and no weights are ever committed**;
the model card is written before any number is claimed.

## 1. Is there an accelerator?

The run is fine-tuned with fp16 on a T4/V100 (Colab) or P100 (Kaggle). On a CPU-only VM it still works — `mixed_precision: auto` resolves to fp32 — but `xlm-roberta-base` takes hours per epoch. The tiny smoke config below is the sanity check: it runs anywhere in seconds.

In [ ]:
import subprocess
import sys

print(
    subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "no nvidia-smi"
)
try:
    import torch

    print("torch", torch.__version__, "cuda", torch.cuda.is_available())
except ImportError:
    print("torch not installed yet — step 2 does that")

## 2. Install

Colab/Kaggle already ship a torch build; `ml[ml,dev]` pulls the data and tracking stack (`numpy`, `scikit-learn`, `pandas`, `mlflow`, `dvc`). The training stack is installed explicitly so the versions are visible in the log.

In [ ]:
!git clone --depth 1 https://github.com/BuildbyArindam/Manovia.git
%cd Manovia
!pip install -q -e "./ml[ml,dev]"
!pip install -q "transformers>=4.44" "peft>=0.11" optuna accelerate datasets
!python -c "import torch, transformers, peft, optuna; print('torch', torch.__version__); print('transformers', transformers.__version__); print('peft', peft.__version__); print('optuna', optuna.__version__)"

## 3. Sanity check first: the tiny smoke run

This trains a 2-layer random-init XLM-R for 12 steps on the committed toy
fixture. It proves *this VM* can run the pipeline end to end — data → LoRA →
thresholds → artefact → evaluation → model card — before an hour of GPU time is
spent on the real thing. Its metrics are noise by construction; the run card
says so.

In [ ]:
!cd ml && python -m manovia_ml.training.emotion.train --config configs/emotion_lora_smoke.yaml --json | tail -40

## 4. Data: GoEmotions, converted to Manovia's nine emotions

Licence and conditions are recorded in `ml/data/CATALOGUE.md` § GoEmotions:
Apache-2.0, **no redistribution of rows**, and remember that it is Reddit text —
the domain shift that every model card for this model has to state.

The first command downloads the three official splits through `datasets`; the
second converts each one with `manovia_ml.training.emotion.data convert`, which
applies the mapping table (`joy`, `sadness`, `anger`, `fear` → themselves,
`nervousness` → `anxiety`, `embarrassment`/`remorse` → `shame`, `grief` →
`sadness`, and so on) and prints what it dropped. **Read that report**: it is
where you see that GoEmotions has no supervision for `loneliness` at all, and
how many rows were dropped for having no mappable label.

In [ ]:
!cd ml && python -m manovia_ml.data.goemotions --download train --split-name train \
    --out data/raw/goemotions/train.jsonl
!cd ml && python -m manovia_ml.data.goemotions --download validation --split-name validation \
    --out data/raw/goemotions/validation.jsonl

In [ ]:
import json
import pathlib
import subprocess

reports = {}
for split in ("train", "validation"):
    source = f"data/raw/goemotions/{split}.jsonl"
    out = f"data/raw/goemotions_multi/{split}.jsonl"
    pathlib.Path("ml/data/raw/goemotions_multi").mkdir(parents=True, exist_ok=True)
    done = subprocess.run(
        [
            sys.executable,
            "-m",
            "manovia_ml.training.emotion.data",
            "convert",
            source,
            "--scheme",
            "goemotions",
            "--lang",
            "en",
            "--out",
            out,
        ],
        cwd="ml",
        capture_output=True,
        text=True,
        check=True,
    )
    reports[split] = json.loads(done.stdout)
    print(
        split,
        "->",
        {
            k: reports[split][k]
            for k in (
                "rows_in",
                "rows_out",
                "rows_with_emotion_target",
                "rows_neutral_only",
                "sha256",
            )
        },
    )
    print("   dropped:", reports[split]["dropped"])
    print(
        "   labels no GoEmotions row can reach:",
        reports[split]["labels_unreachable_from_this_scheme"],
    )

## 5. Version the bytes (never commit them)

DVC records the hashes; the rows stay on the VM. `dvc add` writes a `.dvc` file
that names the bytes without containing them.

In [ ]:
!cd ml && python -m dvc add data/raw/goemotions_multi

## 6. Pin the hashes into a run-local copy of the config

`configs/emotion_lora_xlmr.yaml` ships with `sha256: null` and `rows: null` on
purpose: a config that pins a number nobody measured is worse than one that
refuses to run. Here the pins come straight from the converter's report, into a
**copy** (`configs/_colab_emotion_lora_xlmr.yaml`, git-ignored) so the committed
config stays a template until the owner pastes the real numbers into it.

In [ ]:
import re
import shutil

template = pathlib.Path("ml/configs/emotion_lora_xlmr.yaml")
local = pathlib.Path("ml/configs/_colab_emotion_lora_xlmr.yaml")
text = template.read_text(encoding="utf-8")
for split in ("train", "validation"):
    block = re.search(rf"^  {split}:\n(?:    .*\n)+", text, flags=re.M)
    replacement = (
        f"  {split}:\n"
        f"    path: data/raw/goemotions_multi/{split}.jsonl\n"
        f'    sha256: "{reports[split]["sha256"]}"\n'
        f"    rows: {reports[split]['rows_out']}\n"
    )
    text = text[: block.start()] + replacement + text[block.end() :]
text = text.replace("name: emotion_lora_xlmr", "name: emotion_lora_xlmr", 1)
local.write_text(text, encoding="utf-8")
shutil.copy(local, pathlib.Path("ml/configs/_colab_config_pins.txt"))
print(local, "written")
print(
    "\n".join(
        line
        for line in text.splitlines()
        if "sha256" in line or line.startswith("  train:") or line.startswith("  validation:")
    )
)
print(
    "\n>>> PASTE THESE PINS BACK INTO ml/configs/emotion_lora_xlmr.yaml before quoting any number from this run."
)

## 7. Train

`mixed_precision: auto` → fp16 on CUDA. Expect roughly: 3 epochs over ~43k rows,
batch 32 × 128 tokens, LoRA rank 16 on query/value — comfortably inside a free
T4 session. `--json` prints the run id and the metrics the run logged; the run
itself is in `ml/mlruns` (file store) and holds the params, the data hashes, the
git commit, every metric and the artefact.

In [ ]:
!cd ml && python -m manovia_ml.training.emotion.train --config configs/_colab_emotion_lora_xlmr.yaml --json | tail -60

### Optional: the sweep (≤ 12 trials)

`configs/emotion_lora_xlmr.yaml` has `optuna.enabled: false`; a sweep is a
deliberate act, not a default. Each trial is a full training run (nested MLflow
run), the objective is *validation* macro-F1 at default thresholds — tuning
thresholds inside the sweep would fit them to the split that is choosing the
hyperparameters. On a T4, budget ~1 hour per 12 trials at 1 epoch each.

In [ ]:
# !cd ml && python -m manovia_ml.training.emotion.tune --config configs/_colab_emotion_lora_sweep.yaml --trials 12 --json | tail -20

## 8. Evaluate in a fresh process, from the artefact alone

This is the check that matters for trust: `evaluate.py` refuses to run if the
training module was imported, so the numbers below cannot come from anything but
the saved weights, tokenizer and thresholds. It scores the hand-written
174-row en/hi/bn set (never trained on, never tuned on) with bootstrap 95%
confidence intervals, per-language F1, calibration (ECE) and the error analysis.

In [ ]:
artifact = "artifacts/emotion/manovia-emotion-xlmr-v1"
!cd ml && python -m manovia_ml.training.emotion.evaluate --artifact {artifact} \
    --data data/validation/emotion_validation.jsonl --bootstrap 2000 --json \
    | tee evaluation.json | tail -40

## 9. The comparison a report needs (keyboard baseline + the Day-6 off-the-shelf model)

`report.py` already prints the three-way table: this run against the lexicon keyword chain and against the Day-6 off-the-shelf checkpoint, per language, with intervals — and it flags every language where a baseline beats the fine-tuned model, because that sentence is the one a model card must contain.

In [ ]:
import pathlib

artifact = "artifacts/emotion/manovia-emotion-xlmr-v1"
# The training run writes report.md and MODEL_CARD.md into the artefact.
# Read the card before quoting a number: it is where the honesty lives.
print(pathlib.Path(f"ml/{artifact}/MODEL_CARD.md").read_text(encoding="utf-8")[:3000])

## 10. What to bring back into the repository

Copy these into the repo (they are small; the weights and the rows are **not**):

| bring back | why |
| --- | --- |
| `ml/configs/_colab_emotion_lora_xlmr.yaml`'s two pins | paste `sha256`/`rows` into `configs/emotion_lora_xlmr.yaml` |
| the run id + `ml/mlruns` (or at least the run's JSON artefacts) | a number without a run id is an anecdote |
| `evaluation.json` | per-language F1, CIs, ECE, the top-20 errors |
| `MODEL_CARD.md` from the artefact | goes to `ml/reports/<config>/<run-id>-card.md` and updates `docs/model-card.md` |
| the `report.md` the run wrote | the three-way comparison table |

Then, to use it in the product (never committed):

```bash
EMOTION_ANALYZER=trained
EMOTION_TRAINED_DIR=ml/artifacts/emotion/manovia-emotion-xlmr-v1
```

Deleting that directory is a supported state: the chain logs one warning and the
lexicon answers instead (ADR 0006/0032).

Kaggle notes: the same commands work; `!pip install -q -e "./ml[ml,dev]"` is
enough because Kaggle ships torch, and `%%capture` only if the log is too long.
Colab recycles the VM: `!zip -r mlruns.zip ml/mlruns` before you close the tab.